# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org/) et [PDF](https://python.guillod.org/python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?urlpath=lab/tree/chap08.ipynb).

# 8 Théorie des graphes

<div id="ch:graphes"></div>

Un graphe est un couple $G = (X, E)$ constitué d'un ensemble $X$, non vide et fini, et d'un ensemble $E$ de paires d'éléments de $X$. Les éléments de $X$ sont les sommets du graphe $G$, ceux de $E$ sont les arêtes du graphe $G$.
Un graphe est orienté si les arêtes ont une direction, c'est-à-dire si les couples d'éléments de $E$ sont des listes ordonnées telles que $(i,j) \in E$ n'est pas équivalent à $(j,i)\in E$. Ici seuls des graphes non orientés, c'est-à-dire dont les paires d'éléments de $X$ sont des ensembles non ordonnés ($\{i,j\} \in E$), sont considérés.

Par exemple, le graphe complet à $n$ sommets $K_n$ est le graphe de sommets $X = {1,2,\dots,n}$ ayant comme arêtes les parties à deux éléments de $X$. En particulier, $K_4 = (X,E)$ où $X=\{1,2,3,4\}$ et $E = \big\{ \{1,2\}, \{1,3\}, \{1,4\}, \{2, 3\}, \{2, 4\}, \{3, 4\}  \big\}$.

**Concepts abordés:**

* graphes non orientés

* représentation comme dictionnaire

* utilisation des frozensets

* matrice d'adjacence

* recherche de chemins et de triangles

* fonction récursive

# Exercice 8.1: Graphes comme dictionnaires

Une façon de représenter un graphe $G$ est de le faire avec un dictionnaire dont les clefs sont
les sommets, et la valeur associée à chaque clef $x \in X$ est un ensemble contenant les voisins de $x$.

**a)**
Construire sous forme de dictionnaire les graphes suivants:

<center><img src="https://python.guillod.org/fig/graphes.png" style="width:90%;max-width:800px;"></center>

In [6]:
# Dans les dictionnaires ci-dessous, si i est une clé alors les valeurs attachées sont les sommets s telles que {i,s} est une arrete du graphe

#Graphe (a) à six sommets
a = {1: {2,5}, 2:{1,3,5}, 3:{2,4}, 4:{3,5,6}, 5:{1,2,4}, 6:{4}}
a

{1: {2, 5}, 2: {1, 3, 5}, 3: {2, 4}, 4: {3, 5, 6}, 5: {1, 2, 4}, 6: {4}}

In [7]:
#Graphe (b) : Une 5-clique
b = {}
for i in range(1,6):
    arretes_i = set()
    for j in range(1,6):
        if i!=j:
            arretes_i.add(j)    
    b.update({i :arretes_i})

b

{1: {2, 3, 4, 5},
 2: {1, 3, 4, 5},
 3: {1, 2, 4, 5},
 4: {1, 2, 3, 5},
 5: {1, 2, 3, 4}}

**b)**
Écrire une fonction `complet(n)` qui construit comme dictionnaire le graphe complet $K_n$.

In [8]:
def complet(n):
    K = {}
    for i in range(1,n+1):
        arretes_i = set()
        for j in range(1,n+1):
            if i!=j:
                arretes_i.add(j)    
        K.update({i :arretes_i})
    return K

**c)**
Un graphe donné sous forme de dictionnaire contient l'information plusieurs fois. Écrire une fonction `corriger(graphe)` permettant de rajouter les éléments manquants de sorte que pour tout sommet `x`, si `y` appartient à `graphe[x]`, alors `y` est aussi une clef et `x` appartient à `graphe[y]`. Tester cette fonction.

In [9]:
def corriger(graphe):
    """Permet de rajouter les éléments manquants de sorte que pour tout sommet `x`, si `y` appartient à `graphe[x]`, alors `y` est aussi une clef et `x` appartient à `graphe[y]`

    Args:
        graphe (dict): Graphe sous forme de dictionnaire : chaque clé est un sommet et les valeurs sont les sommets voisins.
    """
    for key in graphe.keys():
        for sommet in graphe[key]:
            if key not in graphe[sommet]:
                graphe[sommet].add(key)
    return

# On teste notre fonction sur la 5-clique de la première question, dont on a volontairement retiré certaines arrêtes

b_alt = {1: {2, 3, 5}, 2: {1, 3, 4, 5}, 3: {1, 2, 4, 5}, 4: {1, 5}, 5: {2, 3}}

corriger(b_alt)

b_alt

{1: {2, 3, 4, 5},
 2: {1, 3, 4, 5},
 3: {1, 2, 4, 5},
 4: {1, 2, 3, 5},
 5: {1, 2, 3, 4}}

**d)**
Écrire une fonction qui retourne l'ensemble (type `set`) de toutes les arêtes d'un graphe représenté par un dictionnaire.

**Indication:**
Les ensembles sont mutables et donc pas hashables.

In [11]:
def aretes(graph):
    aretes = []
    for key in graph.keys():
        for s in graph[key]:
            aretes.append((key, s))
    return set(aretes)

aretes(a)

{(1, 2),
 (1, 5),
 (2, 1),
 (2, 3),
 (2, 5),
 (3, 2),
 (3, 4),
 (4, 3),
 (4, 5),
 (4, 6),
 (5, 1),
 (5, 2),
 (5, 4),
 (6, 4)}

**e)**
<span style="color:red">!</span> Écrire une fonction permettant de déterminer si deux sommets sont connectés par un chemin ou non et qui retourne le chemin si oui.

**Indication:**
Utiliser une fonction récursive.

In [12]:
def connection(graph,a,b,p=[]):
    """Controle si les sommets a et b du graphe 'graph' sont connectés et retourne le chemin parcouru s'il existe

    Args:
        graph (dict): Graphe sous forme de dictionnaire : chaque clé est un sommet et les valeurs sont les sommets voisins.
        a : clé du graphe
        b : clé du graphe
        p : chemin parcouru
    """
    p = p+[a]
    if a==b:
        return p
    for s in graph[a]: # On itère sur les sommets adjacents de a
        p_2 = connection(graph,s,b,p)
        if p_2!=None :
            return p_2
    return None   
    
connection(a,1,6)

RecursionError: maximum recursion depth exceeded in comparison

**f)**
<span style="color:red">!</span> Écrire une fonction qui retourne tous les chemins entre deux sommets (sans les cycles).

# Exercice 8.2: Triangles dans un graphe

Un triangle dans un graphe est un ensemble de trois sommets reliés entre eux par trois arêtes. La recherche et l'analyse des triangles dans un graphe sont importantes pour comprendre sa structure.

**a)**
Déterminer mathématiquement le nombre de sous-ensembles de cardinal trois que possède un ensemble de sommets $X$.

#### Soit n le cardinal de X.
#### Alors on choisit 3 parmis n sommets afin de construire nos triangles.
#### Il y a donc $\binom{n}{3}$ triangles


**b)**
Écrire une fonction retournant l'ensemble des triangles d'un graphe.

In [13]:
def triangles(graph):
    triangles = set()
    for s_0 in graph.keys():
        for s_1 in graph[s_0]:
            for s_2 in graph[s_1]:
                if s_2 in graph[s_0]:
                    triangles.add(frozenset((s_0,s_1,s_2)))
    return triangles

triangles(complet(4))
                    
                
        

{frozenset({1, 2, 3}),
 frozenset({2, 3, 4}),
 frozenset({1, 3, 4}),
 frozenset({1, 2, 4})}

À chaque graphe $G=(X,E)$ correspond une unique matrice $A$ symétrique de taille $n \times n$ avec $n=|X|$ définie par:

$$
A_{ij}=\begin{cases}
1\,, & \text{si}\;\{i,j\}\in E\,,\\ 
0\,, & \text{si}\;\{i,j\}\notin E\,.
\end{cases}
$$

Cette matrice est appelée la matrice d'adjacence du graphe $G$.

**c)**
Définir une fonction retournant la matrice d'adjacence d'un graphe.

**Indication:**
Faire attention que les sommets ne sont pas forcément indexés par des entiers compris entre 0 et $n$ dans le dictionnaire.

In [ ]:
import numpy as np

def adjacence(graph):
    n = len(graph)
    m = np.zeros([n,n])
    # On doit d'abord assigner un indice à chaque sommet du graphe, ce que l'on peut faire avec enumerate
    nouveau_dict = {key: sommet for sommet,key in enumerate(graph.keys())}
    for key in graph.keys():
        for s in graph[key]:
            m[nouveau_dict[key],nouveau_dict[s]] = 1
    return m

In [27]:
# On teste notre code
a = {1: {2,5}, 2:{1,3,5}, 3:{2,4}, 4:{3,5,6}, 5:{1,2,4}, 6:{4}}
adjacence(a)
        
    

array([[0., 1., 0., 0., 1., 0.],
       [1., 0., 1., 0., 1., 0.],
       [0., 1., 0., 1., 0., 0.],
       [0., 0., 1., 0., 1., 1.],
       [1., 1., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0., 0.]])

**d)**
Définir une fonction ayant pour argument une matrice d'adjacence et retournant le graphe correspondant sous forme d'un dictionnaire.

In [30]:
def graphe(matrice):
    """retourne le graphe d'une matrice d'adjacence sous forme d'un dictionnaire

    Args:
        matrice (np.array): matrice d'adjacence
    """
    dictionnaire = {}
    n = len(matrice)
    for i in range(n):
        voisins = set()
        for j in range(n):
            if matrice[i,j] == 1:
                voisins.add(j+1)
        dictionnaire[i+1]=voisins
    return dictionnaire

In [33]:
a = {1: {2,5}, 2:{1,3,5}, 3:{2,4}, 4:{3,5,6}, 5:{1,2,4}, 6:{4}}
m_a = adjacence(a)

graphe(m_a)

{1: {2, 5}, 2: {1, 3, 5}, 3: {2, 4}, 4: {3, 5, 6}, 5: {1, 2, 4}, 6: {4}}

**e)**
En utilisant la matrice d'adjacence $A$ et la matrice $B=A^2$, écrire une fonction retournant l'ensemble des triangles d'un graphe.

In [41]:
a = {1: {2,5}, 2:{1,3,5}, 3:{2,4}, 4:{3,5,6}, 5:{1,2,4}, 6:{4}}
m_a@m_a@m_a

array([[2., 4., 2., 2., 4., 1.],
       [4., 2., 5., 1., 6., 2.],
       [2., 5., 0., 5., 1., 0.],
       [2., 1., 5., 0., 6., 3.],
       [4., 6., 1., 6., 2., 0.],
       [1., 2., 0., 3., 0., 0.]])

Le nombre de triangles est donné par le nombre de 3-cycles, on peut donc prendre les coefficients diagonaux (i,i) de A^3 lesquels correspondent aux nombre de 3-cycles partant du sommet i du graphe. Comme il peut être parcouru dans les deux sens, et que l'on peut parcourir un triangle en partant de chacun des trois sommets, la trace de A^3 est égale à 6 fois le nombre de triangles du graphe.



In [47]:
def triangles(graph):
    m = adjacence(graph)
    m_3 = np.linalg.matrix_power(m, 3)
    return np.trace(m_3)/6

triangles(a)

1.0

**f)**
En utilisant la matrice d'adjacence $A$, écrire une fonction calculant le nombre de triangles.

**Indication:**
Interpréter les entrées de la matrice $A^n$.

In [48]:
triangles(complet(10))

120.0

# Exercice 8.3: <span style="color:red">!!</span> Module NetworkX

De nombreux algorithmes de la théorie des graphes sont implémentés dans le module networkX, voir la documentation [ici](https://networkx.org/documentation/).

**a)**
Suivre le tutoriel de networkX disponible [ici](https://networkx.org/documentation/stable/tutorial.html).

**b)**
Analyser un des graphes téléchargeables à l'adresse: <https://github.com/gephi/gephi/wiki/Datasets> ou <https://snap.stanford.edu/data/>.